<a href="https://colab.research.google.com/github/arthurflor23/handwritten-text-recognition/blob/master/tutorial.ipynb" target="_parent"><img src="https://img.shields.io/badge/Colab-F9AB00?style=for-the-badge&logo=googlecolab&color=525252" alt="Open In Colab"/></a>
<a href="https://github.com/arthurflor23/handwritten-text-recognition" target="_parent"><img src="https://img.shields.io/badge/GitHub-Repository-181717?style=for-the-badge&logo=github" alt="GitHub Repository"/></a>
<a href="https://ko-fi.com/arthurflor23" target="_parent"><img src="https://img.shields.io/badge/Ko--fi-F16061?style=for-the-badge&logo=ko-fi&logoColor=white" alt="Support on Ko-fi"/></a>

# Handwritten Text Recognition with PySarah (Google Colab Edition)

## 1. Introduction
In this tutorial, we will explore **[PySarah](https://github.com/arthurflor23/handwritten-text-recognition)**, an end-to-end framework for handwriting recognition and text-to-image synthesis. 

This notebook is specifically configured for **Google Colab** with free NVIDIA GPU acceleration (T4/V100/A100).

## 2. Colab Environment & GPU Check

First, make sure your Colab runtime is set to **GPU**:
- Go to menu: **Runtime** -> **Change runtime type** -> Select **T4 GPU** (or any GPU) -> Click **Save**.

Let's verify that the GPU is available:

In [ ]:
# Check NVIDIA GPU status
!nvidia-smi

In [ ]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {gpus[0].name}")
else:
    print("⚠️ No GPU detected. Please change runtime type to GPU for fast training.")

## 3. Setup Project in Colab

You have two convenient ways to load the project in Google Colab:
- **Option A (Zero Upload - Direct Git Clone)**: Recommended! Colab clones the repo directly from GitHub.
- **Option B (Upload ZIP)**: If you packaged your local folder as `handwritten-text-recognition-colab.zip` (18 MB), upload it via Colab's left sidebar Files tab.

In [ ]:
# OPTION A: Direct Git Clone (Recommended)
import os

if not os.path.exists('/content/handwritten-text-recognition'):
    !git clone https://github.com/arthurflor23/handwritten-text-recognition.git /content/handwritten-text-recognition

%cd /content/handwritten-text-recognition

# Install dependencies (Colab already has TensorFlow GPU pre-installed)
!pip install -q mlflow fonttools levenshtein openai opencv-python-headless pandas gdown

In [ ]:
# Apply Keras 3 compatibility patch to flor.py if needed
import re

flor_file = 'sarah/models/recognition/flor.py'
if os.path.exists(flor_file):
    with open(flor_file, 'r', encoding='utf-8') as f:
        content = f.read()
    # Remove obsolete use_gate=True for Keras 3
    if 'use_gate=True' in content:
        content = re.sub(r'use_gate=True,?\s*', '', content)
        with open(flor_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ Patched flor.py for Keras 3 compatibility.")
    else:
        print("✅ flor.py is already compatible.")

In [ ]:
# OPTION B: Mount Google Drive & Unzip project archive
from google.colab import drive
drive.mount('/content/drive')

# Path to the zip file in your Google Drive (adjust if inside a folder, e.g. MyDrive/Colab Notebooks/...)
drive_zip = '/content/drive/MyDrive/handwritten-text-recognition-colab.zip'

# Unzip directly into Colab's fast local SSD storage (/content/)
!unzip -q -o "$drive_zip" -d /content/handwritten-text-recognition

%cd /content/handwritten-text-recognition

# Install dependencies
!pip install -q mlflow fonttools levenshtein openai opencv-python-headless pandas gdown


## 4. Download English Handwriting Datasets

PySarah supports 4 major English handwriting datasets:
1. **`washington`** *(Recommended to start)*: 18th-century cursive English letters by George Washington (656 lines, ~20 MB).
2. **`iam`**: Gold standard for modern English handwriting from 500+ writers (13,353 lines, ~350 MB).
3. **`bentham`**: 18th–19th century historical English manuscripts by philosopher Jeremy Bentham (~11,500 lines, ~250 MB).
4. **`cvl-database`**: Modern European handwriting in English by 310 writers (~10,000 lines, ~400 MB).

Run this cell to download the English datasets you want (select `washington`, `iam`, `bentham`, or `cvl-database`):

In [ ]:
# Download English datasets directly in Colab
# Choose one or multiple English datasets: ['washington', 'iam', 'bentham', 'cvl-database']
english_datasets_to_download = ['washington']  # Add 'iam', 'bentham', 'cvl-database' as needed

for ds_name in english_datasets_to_download:
    !python download_datasets.py --download {ds_name}

print("\nAll requested English datasets are ready!")


## 5. Dataset & Tokenizer Initialization

Now let's load the dataset using PySarah's `Dataset` class:
- `text_level='line'`: Line-level recognition
- `image_shape=(64, 1024, 1)`: Normalized image resolution (64px height, 1024px width, 1 grayscale channel)
- `tokenizer`: Handles converting text characters into numeric arrays and vice-versa.

In [ ]:
from sarah import Dataset, Compose

seed = 42
image_shape = (64, 1024, 1)
recognition = 'flor'

# 1. Initialize tokenizer
tokenizer, run_context = Compose.get_tokenizer(recognition=recognition, recognition_run_id=None)

# 2. Choose your active English source:
# Options: 'washington', 'iam', 'bentham', 'cvl-database', or 'all-english' (combines all downloaded English datasets)
active_english_source = 'washington'

dataset = Dataset(source=active_english_source,
                  text_level='line',
                  image_shape=image_shape,
                  fonts_path='fonts',
                  tokenizer=tokenizer,
                  seed=seed)

# Display dataset & tokenizer summary cards
dataset


## 6. Augmentor Class (Data Augmentation)

Real handwriting contains natural variations in slant, pen pressure, and stroke width.
The `Augmentor` applies realistic distortions dynamically during training so the neural network learns to generalize:

In [ ]:
from sarah import Augmentor

augmentor = Augmentor(mixup=None,
                      erode=(0.33, 3),
                      dilate=None,
                      elastic=None,
                      perspective=(0.66, 0.3),
                      shear=(0.33, 0.3),
                      rotate=(0.33, 0.3),
                      scale=(0.33, 0.1),
                      shift_y=None,
                      shift_x=None,
                      salt_and_pepper=None,
                      gaussian_noise=None,
                      gaussian_blur=None,
                      seed=seed)
augmentor

## 7. Compose Class & Model Architecture

The `Compose` class orchestrates the entire pipeline:
- Builds the neural network model (`flor`: Residual Gated CNN + Bidirectional LSTM + Multi-Head Attention)
- Compiles the optimizer (AdamW) and Connectionist Temporal Classification (CTC) loss
- Integrates MLflow experiment tracking

In [ ]:
from sarah import Compose

compose = Compose(recognition=recognition,
                  image_shape=image_shape,
                  tokenizer=dataset.tokenizer,
                  seed=seed)

compose.compile(learning_rate=1e-3, run_context=run_context)
compose

## 8. Model Training (GPU Accelerated)

With Google Colab's GPU, training runs at full speed! 
Feel free to adjust `epochs` (e.g. 50 or 100 for a solid model, or 1000 for full convergence).

In [ ]:
epochs = 50
batch_size = 16
plateau_factor = 0.1
plateau_cooldown = 0
plateau_patience = 15
patience = 25

training_gen, training_steps = dataset.get_generator(data_partition='training',
                                                     batch_size=batch_size,
                                                     augmentor=augmentor,
                                                     shuffle=True)
validation_gen, validation_steps = dataset.get_generator(data_partition='validation',
                                                         batch_size=batch_size)

compose.fit(training_gen=training_gen,
            training_steps=training_steps,
            validation_gen=validation_gen,
            validation_steps=validation_steps,
            plateau_factor=plateau_factor,
            plateau_cooldown=plateau_cooldown,
            plateau_patience=plateau_patience,
            patience=patience,
            epochs=epochs)

## 9. Model Prediction

Now let's use the trained model to decode handwritten text from the test partition:

In [ ]:
top_paths = 1
beam_width = 32

test_gen, test_steps = dataset.get_generator(data_partition='test',
                                             batch_size=16)

predictions, probabilities = compose.predict_recognition(x=test_gen,
                                                         steps=test_steps,
                                                         top_paths=top_paths,
                                                         beam_width=beam_width,
                                                         ctc_decode=True,
                                                         token_decode=True,
                                                         verbose=1)

print(f"\nGenerated predictions for {len(predictions)} samples.")


## 10. Model Evaluation (Character & Word Error Rates)

We evaluate model performance using standard metrics:
- **Character Error Rate (CER)**: Levenshtein edit distance at character level.
- **Word Error Rate (WER)**: Edit distance at word level.

In [ ]:
import json

source_gen, source_steps = dataset.get_generator(data_partition='test',
                                                 batch_size=16,
                                                 batch_encoded=False)

metrics, evaluations = compose.evaluate_recognition(x=predictions,
                                                    y=source_gen,
                                                    steps=source_steps,
                                                    probabilities=probabilities,
                                                    verbose=1)

print("Evaluation Metrics:", metrics)
print(json.dumps(evaluations['data'], indent=4, ensure_ascii=False)[:1000] + '...\n(truncated)')


## 11. Visualizing Test Predictions

Finally, let's plot a few test images and display the ground truth alongside the model's predicted recognition:

In [ ]:
import matplotlib.pyplot as plt

num_samples = 5

# 1. Generator for raw unencoded ground truth text
src_gen, _ = dataset.get_generator(data_partition='test',
                                   batch_size=num_samples,
                                   batch_encoded=False,
                                   batch_processing=False,
                                   batch_scale=False,
                                   shuffle=False)

# 2. Generator for decoded image numpy arrays
enc_gen, _ = dataset.get_generator(data_partition='test',
                                   batch_size=num_samples,
                                   batch_encoded=True,
                                   batch_processing=False,
                                   batch_scale=False,
                                   shuffle=False)

_, y_src = next(src_gen)
true_texts = y_src[1]

_, y_enc = next(enc_gen)
images = y_enc[0]

# 3. Plot test images with True vs Predicted text
fig, axes = plt.subplots(nrows=len(images), figsize=(14, 2.5 * len(images)))
if len(images) == 1:
    axes = [axes]

for i, (img, true_text) in enumerate(zip(images, true_texts)):
    pred = predictions[i] if i < len(predictions) else 'N/A'
    pred_text = pred[0] if isinstance(pred, (list, tuple)) else pred
    
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"True: {true_text}\nPred: {pred_text}", fontsize=11, loc='left')
    axes[i].axis('off')

plt.tight_layout()
plt.show()


## 12. Continuous Multi-Dataset Runner (All Datasets)

This section automatically loops through **all available datasets**, downloads them, trains a dedicated model for each, evaluates test accuracy, and displays predictions continuously in one go.

### Dataset Types in PySarah:
1. **Line-Level Sequence Datasets (64×1024)**: 
   - `washington`: 18th-century English cursive letters
   - `saintgall`: 9th-century Latin medieval manuscript
   - `parzival`: 13th-century Middle High German epic
   - `cvl-digits`: Modern handwritten number strings
   - `orand-car-a`: Bank check courtesy amount digit strings
   - `iam`: Modern English handwriting benchmark (500+ writers)
2. **Single-Character Datasets (28×28)**: 
   - `mnist`: Isolated single digits (0–9)
   - `emnist`: Isolated single characters (A–Z, a–z, 0–9)


In [ ]:
# CELL 1: Define Dataset Configurations & Target Datasets to Run
LINE_DATASETS = {
    'washington':   {'desc': '18th-century English cursive', 'download_key': 'washington'},
    'saintgall':    {'desc': '9th-century Latin manuscript',  'download_key': 'saintgall'},
    'parzival':     {'desc': '13th-century German manuscript', 'download_key': 'parzival'},
    'cvl-digits':   {'desc': 'Handwritten number strings',    'download_key': 'cvl-digits'},
    'orand-car-a':  {'desc': 'Bank check amount strings',     'download_key': 'orand-car'},
    'iam':          {'desc': 'Modern English handwriting',     'download_key': 'iam'}
}

# Select which datasets you want to run in the continuous pipeline:
# (By default, runs the 4 immediate benchmarks; add 'orand-car-a' or 'iam' as desired)
active_line_datasets = ['washington', 'saintgall', 'parzival', 'cvl-digits']

print(f"Selected {len(active_line_datasets)} datasets for continuous training and evaluation:")
for k in active_line_datasets:
    print(f"  - {k}: {LINE_DATASETS[k]['desc']}")


In [ ]:
# CELL 2: Continuous Multi-Dataset Training, Prediction, & Evaluation Loop
import os
import pandas as pd
import matplotlib.pyplot as plt
from sarah import Dataset, Augmentor, Compose

seed = 42
image_shape = (64, 1024, 1)
recognition = 'flor'
epochs = 20          # Adjust epochs per dataset (15-30 recommended for fast demo)
batch_size = 16

augmentor = Augmentor(erode=(0.33, 3), perspective=(0.66, 0.3), shear=(0.33, 0.3), rotate=(0.33, 0.3), scale=(0.33, 0.1), seed=seed)

benchmark_summary = []
trained_models = {}

for ds_name in active_line_datasets:
    dl_key = LINE_DATASETS[ds_name]['download_key']
    print("\n" + "="*75)
    print(f"  STARTING PIPELINE FOR: {ds_name.upper()} ({LINE_DATASETS[ds_name]['desc']})")
    print("="*75)
    
    # 1. Download if missing
    !python download_datasets.py --download {dl_key}
    
    # 2. Load Dataset & Tokenizer
    print(f"\n[1/4] Loading {ds_name} dataset...")
    tokenizer, run_context = Compose.get_tokenizer(recognition=recognition, recognition_run_id=None)
    ds = Dataset(source=ds_name, text_level='line', image_shape=image_shape, fonts_path='fonts', tokenizer=tokenizer, seed=seed)
    
    n_train = len(ds.samples['source']['training'])
    n_val = len(ds.samples['source']['validation'])
    n_test = len(ds.samples['source']['test'])
    print(f"      Loaded: {n_train} train | {n_val} val | {n_test} test samples.")
    
    # 3. Train Model
    print(f"\n[2/4] Training dedicated model for {ds_name} ({epochs} epochs)... ")
    model = Compose(recognition=recognition, image_shape=image_shape, tokenizer=ds.tokenizer, seed=seed)
    model.compile(learning_rate=1e-3, run_context=run_context)
    
    train_gen, train_steps = ds.get_generator('training', batch_size=batch_size, augmentor=augmentor, shuffle=True)
    val_gen, val_steps = ds.get_generator('validation', batch_size=batch_size)
    
    model.fit(training_gen=train_gen, training_steps=train_steps, validation_gen=val_gen, validation_steps=val_steps,
              plateau_factor=0.1, plateau_patience=10, patience=15, epochs=epochs)
    trained_models[ds_name] = model
    
    # 4. Predict on Test Set
    print(f"\n[3/4] Predicting on {ds_name} test partition...")
    test_gen, test_steps = ds.get_generator('test', batch_size=batch_size)
    preds, probs = model.predict_recognition(x=test_gen, steps=test_steps, beam_width=32, top_paths=1, verbose=0)
    
    # 5. Evaluate Metrics (CER / WER)
    src_gen, src_steps = ds.get_generator('test', batch_size=batch_size, batch_encoded=False)
    metrics, _ = model.evaluate_recognition(x=preds, y=src_gen, steps=src_steps, probabilities=probs, verbose=0)
    
    cer = metrics.get('cer', 0.0)
    wer = metrics.get('wer', 0.0)
    print(f"      Results: CER = {cer:.4f} | WER = {wer:.4f}")
    benchmark_summary.append({
        'Dataset': ds_name,
        'Description': LINE_DATASETS[ds_name]['desc'],
        'Test Lines': n_test,
        'CER': f"{cer:.4f}",
        'WER': f"{wer:.4f}"
    })
    
    # 6. Immediate Visual Samples
    print(f"\n[4/4] Visual Test Predictions for {ds_name}:")
    viz_src, _ = ds.get_generator('test', batch_size=2, batch_encoded=False, batch_processing=False, batch_scale=False, shuffle=False)
    viz_enc, _ = ds.get_generator('test', batch_size=2, batch_encoded=True, batch_processing=False, batch_scale=False, shuffle=False)
    _, y_src = next(viz_src)
    _, y_enc = next(viz_enc)
    
    true_texts = y_src[1]
    images = y_enc[0]
    
    fig, axes = plt.subplots(nrows=len(images), figsize=(12, 2.2 * len(images)))
    if len(images) == 1:
        axes = [axes]
    for i, (img, true_text) in enumerate(zip(images, true_texts)):
        p = preds[i] if i < len(preds) else 'N/A'
        p_text = p[0] if isinstance(p, (list, tuple)) else p
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"[{ds_name}] True: {true_text}\nPred: {p_text}", fontsize=10, loc='left')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

print("\n" + "="*75)
print("               FINAL MULTI-DATASET BENCHMARK SUMMARY")
print("="*75)
display(pd.DataFrame(benchmark_summary))


### 12.1. Isolated Single Characters: MNIST & EMNIST

`mnist` and `emnist` are **isolated single character datasets** (28×28 pixels), unlike line-level handwriting (64×1024).
They use `text_level='char'` and `image_shape=(28, 28, 1)`:

In [ ]:
# Load MNIST (Isolated Digits)
!python download_datasets.py --download mnist

mnist_dataset = Dataset(source='mnist',
                        text_level='char',
                        image_shape=(28, 28, 1),
                        fonts_path='fonts',
                        seed=42)
mnist_dataset
